#  IA Emocional y Cognitiva con PyTorch
### Clase de Deep Learning — Ejemplo Práctico Integrado

| Concepto | ¿Qué hace? | Arquitectura |
|---|---|---|
| **IA Emocional** | Detecta emociones en rostros en tiempo real | Vision Transformer (ViT) fine-tuned en FER-2013 |
| **IA Cognitiva** | Razona y genera una respuesta contextual | LLM (TinyLlama 1.1B Chat) |

### Flujo del modelo:
```
 Webcam ──►  Haar Cascade  ──►  ViT Emotion  ──►  TinyLlama
            (detecta rostro)     (clasifica emoción)   (genera consejo)
```


In [4]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   
os.environ["TOKENIZERS_PARALLELISM"]  = "false" 
os.environ["CUDA_LAUNCH_BLOCKING"]    = "1"     

import torch
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")  
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings("ignore")

from PIL import Image
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    AutoTokenizer,
    AutoModelForCausalLM
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 55)
print(f"    Dispositivo     : {DEVICE.upper()}")
if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"   GPU             : {gpu_name}")
else:
    print("   No se encontro CUDA. Modo CPU .")
print("=" * 55)


    Dispositivo     : CUDA
   GPU             : NVIDIA GeForce RTX 3050 6GB Laptop GPU


---
## IA Emocional: Reconocimiento de Emociones Faciales

### ¿Qué es la IA Emocional?
La IA Emocional (también llamada *Affective Computing*) busca que las máquinas **detecten, interpreten y respondan** a las emociones humanas.




In [ ]:
# Cargar modelo ViT de emociones
EMOTION_MODEL_NAME = "trpakov/vit-face-expression"

print("Descargando/cargando ViT de emociones...")
print("  (primera vez puede tardar ~1 min)")

em_processor = AutoImageProcessor.from_pretrained(EMOTION_MODEL_NAME)
em_model     = AutoModelForImageClassification.from_pretrained(EMOTION_MODEL_NAME)
em_model     = em_model.to(DEVICE)
em_model.eval()

EMOCIONES_ES = {
    "angry"   : "Enojado",
    "disgust" : "Disgustado",
    "fear"    : "Asustado",
    "happy"   : "Feliz",
    "neutral" : "Neutral",
    "sad"     : "Triste",
    "surprise": "Sorprendido"
}

print("  Ejecutando warmup CUDA...")
with torch.no_grad():
    _dummy_img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    _dummy_in  = em_processor(images=_dummy_img, return_tensors="pt").to(DEVICE)
    _          = em_model(**_dummy_in)
    del _dummy_img, _dummy_in, _
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print("  Warmup completado.")

print(f"\nModelo ViT cargado en {DEVICE.upper()}")
print(f"  Arquitectura : {em_model.config.model_type}")
print(f"  Clases       : {list(em_model.config.id2label.values())}")


Descargando/cargando ViT de emociones...
  (primera vez puede tardar ~1 min)
  Ejecutando warmup CUDA...
  Warmup completado.

Modelo ViT cargado en CUDA
  Arquitectura : vit
  Clases       : ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [ ]:
import subprocess, sys, os, tempfile

WEBCAM_HELPER = os.path.join(os.getcwd(), "webcam_capture.py")

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def capturar_webcam_seguro(camara=0, timeout=10):
    """
    Captura via proceso separado para proteger el kernel de Jupyter.
    Si el driver de camara da segfault, solo muere el subprocess.
    """
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    tmp.close()
    out_path = tmp.name
    try:
        result = subprocess.run(
            [sys.executable, WEBCAM_HELPER, out_path, str(camara)],
            timeout=timeout, capture_output=True, text=True
        )
        if result.returncode == 0 and os.path.exists(out_path):
            frame = cv2.imread(out_path)
            try: os.unlink(out_path)
            except: pass
            return (frame, None) if frame is not None else (None, "imread devolvio None")
        err = result.stderr.strip() or "proceso sin output"
        try: os.unlink(out_path)
        except: pass
        return None, err
    except subprocess.TimeoutExpired:
        try: os.unlink(out_path)
        except: pass
        return None, f"Timeout ({timeout}s)"
    except Exception as e:
        try: os.unlink(out_path)
        except: pass
        return None, str(e)

def crear_imagen_prueba():
    frame = np.zeros((480, 640, 3), dtype=np.uint8)
    frame[:] = (20, 20, 40)
    cv2.circle(frame, (320, 220), 120, (210, 185, 160), -1)
    cv2.ellipse(frame, (275, 195), (22, 15), 0, 0, 360, (40, 40, 40), -1)
    cv2.ellipse(frame, (365, 195), (22, 15), 0, 0, 360, (40, 40, 40), -1)
    cv2.circle(frame, (278, 196), 8, (10, 10, 10), -1)
    cv2.circle(frame, (368, 196), 8, (10, 10, 10), -1)
    cv2.ellipse(frame, (320, 265), (55, 25), 0, 0, 180, (40, 40, 40), 3)
    cv2.putText(frame, "Sin camara - imagen de prueba",
                (70, 420), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (150, 200, 255), 2)
    return frame

def detectar_emocion(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
    )
    if len(faces) == 0:
        return None, None, None, None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad = 15
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    x1 = max(0, x - pad);  y1 = max(0, y - pad)
    x2 = min(frame.shape[1], x + w + pad)
    y2 = min(frame.shape[0], y + h + pad)
    rostro_pil = Image.fromarray(frame_rgb[y1:y2, x1:x2])


    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    for device_try in ([DEVICE, "cpu"] if DEVICE == "cuda" else ["cpu"]):
        try:
            with torch.no_grad():
                inputs = em_processor(
                    images=rostro_pil, return_tensors="pt"
                ).to(device_try)
                em_model.to(device_try)
                outputs = em_model(**inputs)
                probs   = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()

            if device_try != DEVICE:
                em_model.to(DEVICE)
                print(f"  (inferencia en CPU — GPU no disponible momentaneamente)")
            idx_max    = probs.argmax()
            emocion_en = em_model.config.id2label[idx_max]
            confianza  = float(probs[idx_max])
            return emocion_en, confianza, (x, y, w, h), probs
        except Exception as e:
            print(f"  Fallo en {device_try}: {e}")
            if device_try == "cpu":
                return None, None, None, None
            continue

print("Funciones cargadas correctamente.")
print(f"  Helper webcam : {WEBCAM_HELPER}")
print(f"  Existe        : {os.path.exists(WEBCAM_HELPER)}")


Funciones cargadas correctamente.
  Helper webcam : c:\Users\diego\Desktop\laboratorio\webcam_capture.py
  Existe        : True


In [ ]:
plt.close("all")

print("Capturando imagen...")
frame, err = capturar_webcam_seguro(camara=0, timeout=10)

if frame is None:
    print(f"  Camara no disponible ({err}). Usando imagen de prueba.")
    frame = crear_imagen_prueba()
else:
    print("  Imagen capturada correctamente.")

print("Detectando emocion...")
emocion_en, confianza, bbox, todas_probs = detectar_emocion(frame)

if emocion_en is None:
    print("  No se detecto ningun rostro. Prueba acercarte mas a la camara.")
else:
    emocion_es = EMOCIONES_ES.get(emocion_en, emocion_en)
    print(f"  Emocion detectada: {emocion_es} ({confianza*100:.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0d1117")
    fig.suptitle("IA EMOCIONAL - Deteccion de Emocion Facial",
                 color="white", fontsize=14, fontweight="bold", y=1.01)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    axes[0].imshow(frame_rgb)
    axes[0].set_facecolor("#0d1117")
    if bbox is not None:
        x, y_pos, w, h = bbox
        rect = patches.Rectangle(
            (x, y_pos), w, h, linewidth=3, edgecolor="#00ff88", facecolor="none"
        )
        axes[0].add_patch(rect)
        axes[0].text(x, y_pos - 12,
                     f"{emocion_es}  {confianza*100:.0f}%",
                     fontsize=12, color="#00ff88", fontweight="bold",
                     bbox=dict(facecolor="black", alpha=0.75, edgecolor="#00ff88"))
    axes[0].set_title("Frame capturado", color="#aaaaaa", fontsize=11)
    axes[0].axis("off")

    labels_en  = [em_model.config.id2label[i] for i in range(len(todas_probs))]
    labels_vis = [EMOCIONES_ES.get(l, l) for l in labels_en]
    colores    = ["#00ff88" if l == emocion_en else "#3a3a5c" for l in labels_en]
    bars = axes[1].barh(labels_vis, todas_probs * 100,
                        color=colores, edgecolor="none", height=0.6)
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel("Confianza (%)", color="#aaaaaa")
    axes[1].set_title("Probabilidades por emocion", color="#aaaaaa", fontsize=11)
    axes[1].set_facecolor("#161b22")
    axes[1].tick_params(colors="white")
    for spine in axes[1].spines.values():
        spine.set_color("#444")
    for bar, prob in zip(bars, todas_probs):
        axes[1].text(prob * 100 + 1, bar.get_y() + bar.get_height() / 2,
                     f"{prob*100:.1f}%", va="center", color="white", fontsize=9)

    plt.tight_layout()
    plt.savefig("demo_emocional.png", dpi=100, bbox_inches="tight",
                facecolor="#0d1117")  # guardar a archivo (mas estable que plt.show)
    plt.show()
    plt.close(fig)

    sep = "=" * 50
    print(f"\n{sep}")
    print(f"  Emocion detectada : {emocion_es}")
    print(f"  Confianza         : {confianza*100:.1f}%")
    print(f"  Modelo utilizado  : ViT fine-tuned en FER-2013")
    print(f"{sep}")


Capturando imagen...
  Imagen capturada correctamente.
Detectando emocion...
  Emocion detectada: Asustado (62.1%)

  Emocion detectada : Asustado
  Confianza         : 62.1%
  Modelo utilizado  : ViT fine-tuned en FER-2013


---
## PARTE 2 — IA Cognitiva: Razonamiento y Respuesta Contextual

### ¿Qué es la IA Cognitiva?

### ¿Cómo lo hacemos?
Usamos un **LLM (Large Language Model)** pequeño: **TinyLlama 1.1B**

```
Emoción detectada → Prompt estructurado → TinyLlama → Respuesta empática
     "Triste"      → [system + user msg] →  LLM 1.1B → "Entiendo cómo te sientes..."
```

### TinyLlama vs los LLMs grandes
| Modelo | Parámetros | VRAM necesaria | Velocidad |
|---|---|---|---|
| **TinyLlama 1.1B** | **1.1 B** | **~2.5 GB** | **RTX** |


In [ ]:

LLM_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Descargando/cargando TinyLlama...")

if DEVICE == "cuda":
    torch.cuda.empty_cache()

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model     = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16,  # precision media -> mitad VRAM
    device_map="auto"
)
llm_model.eval()

if DEVICE == "cuda":
    vram_usada = torch.cuda.memory_allocated() / 1024**3
    print(f"\nTinyLlama cargado")
    print(f"  Parametros : ~1.1 Billones")
    print(f"  Precision  : float16")
    print(f"  VRAM usada : {vram_usada:.2f} GB")
else:
    print("\nTinyLlama cargado en CPU")


Descargando/cargando TinyLlama...

TinyLlama cargado
  Parametros : ~1.1 Billones
  Precision  : float16
  VRAM usada : 2.38 GB


In [ ]:
def generar_respuesta_cognitiva(emocion_es: str, max_tokens: int = 60) -> str:

    system_prompt = (
        "Eres un asistente de inteligencia artificial empático. "
        "Tu tarea es dar un consejo claro basado en la emoción detectada. "
        "Reglas estrictas: "
        "1) Responde en español, "
        "2) EXACTAMENTE 2 oraciones, "
        "3) Sé claro, coherente y directo, "
        "4) No uses listas ni viñetas, "
        "5) Da solo un consejo práctico."
    )

    user_prompt = (
        f"La emoción detectada del usuario es: {emocion_es}. "
        f"Genera una respuesta empática y útil."
    )

    prompt = (
        f"<|system|>\n{system_prompt}</s>\n"
        f"<|user|>\n{user_prompt}</s>\n"
        f"<|assistant|>\n"
    )

    inputs = llm_tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.5,   
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    new_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    respuesta = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    respuesta = respuesta.split("\n")[0]  # evita basura extra

    partes = respuesta.split(".")
    if len(partes) > 2:
        respuesta = ".".join(partes[:2]) + "."

    if len(respuesta) < 10:
        if "triste" in emocion_es.lower():
            respuesta = "Parece que estás pasando un momento difícil. Te recomiendo descansar un poco y hacer algo que te relaje."
        elif "feliz" in emocion_es.lower():
            respuesta = "Se nota que estás en un buen momento. Aprovecha esta energía para seguir con lo que te gusta."
        else:
            respuesta = "Gracias por compartir tu estado. Te recomiendo tomarte un momento para reflexionar y cuidarte."

    return respuesta


print(" Función cognitiva optimizada y lista")

 Función cognitiva optimizada y lista


In [ ]:
import textwrap
plt.close("all")

emociones_prueba = ["Feliz", "Triste", "Enojado", "Asustado"]

COLORES_EMOCION = {
    "Feliz"      : "#00ff88",
    "Triste"     : "#4488ff",
    "Enojado"    : "#ff4444",
    "Asustado"   : "#ffaa00",
    "Neutral"    : "#aaaaaa",
    "Sorprendido": "#ff88ff",
    "Disgustado" : "#88ff44"
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor("#0d1117")
fig.suptitle("IA COGNITIVA - Razonamiento por Emocion",
             color="white", fontsize=14, fontweight="bold")

for ax, emocion in zip(axes.flatten(), emociones_prueba):
    print(f"Generando respuesta para: {emocion}...", end=" ", flush=True)
    respuesta = generar_respuesta_cognitiva(emocion)
    print("OK")
    color = COLORES_EMOCION.get(emocion, "#ffffff")
    ax.set_facecolor("#161b22")
    ax.axis("off")

    ax.text(0.5, 0.92, f"Emocion: {emocion}", transform=ax.transAxes,
            ha="center", va="top", fontsize=13, fontweight="bold", color=color)

    ax.plot([0.05, 0.95], [0.85, 0.85], transform=ax.transAxes,
            color=color, linewidth=1, alpha=0.5)

    wrapped = "\n".join(textwrap.wrap(respuesta, width=55))
    ax.text(0.5, 0.80, wrapped, transform=ax.transAxes,
            ha="center", va="top", fontsize=9.5, color="#dddddd",
            bbox=dict(facecolor="#0d1117", edgecolor="none", alpha=0.5))

plt.tight_layout()
plt.savefig("demo_cognitivo.png", dpi=100, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()
plt.close(fig)


Generando respuesta para: Feliz... OK
Generando respuesta para: Triste... OK
Generando respuesta para: Enojado... OK
Generando respuesta para: Asustado... OK


---
##  DEMO COMPLETO — IA Emocional + Cognitiva Integradas

Aquí conectamos ambas partes:  
Tu cámara → detecta emoción → genera respuesta personalizada


In [ ]:
import textwrap

def demo_completo():
    """Demo integrado: webcam -> emocion -> respuesta cognitiva."""
    plt.close("all")
    print("=" * 60)
    print("  DEMO COMPLETO - IA Emocional + Cognitiva")
    print("=" * 60)

    # ETAPA 1: Captura segura
    print("\n[1] Capturando imagen...")
    frame, err = capturar_webcam_seguro(camara=0, timeout=10)
    if frame is None:
        print(f"  Camara no disponible ({err}). Usando imagen sintetica.")
        frame = crear_imagen_prueba()
    else:
        print("  Imagen capturada OK.")

    # ETAPA 2: IA Emocional
    print("[2] IA Emocional - Analizando rostro...")
    emocion_en, confianza, bbox, todas_probs = detectar_emocion(frame)
    if emocion_en is None:
        print("  No se detecto rostro.")
        return
    emocion_es = EMOCIONES_ES.get(emocion_en, emocion_en)
    print(f"  -> {emocion_es}  ({confianza*100:.1f}%)")

    # ETAPA 3: IA Cognitiva
    print("[3] IA Cognitiva - Generando respuesta...")
    emocion_nombre = emocion_es.strip().split()[-1]
    respuesta = generar_respuesta_cognitiva(emocion_nombre)
    print("  -> Respuesta OK.")

    fig = plt.figure(figsize=(16, 9))
    fig.patch.set_facecolor("#0d1117")
    gs       = fig.add_gridspec(2, 2, width_ratios=[1.4, 1], hspace=0.4, wspace=0.3)
    ax_img   = fig.add_subplot(gs[:, 0])
    ax_bars  = fig.add_subplot(gs[0, 1])
    ax_text  = fig.add_subplot(gs[1, 1])

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    ax_img.imshow(frame_rgb)
    ax_img.set_facecolor("#0d1117")
    ax_img.axis("off")
    if bbox is not None:
        x, y_pos, w, h = bbox
        c_box = "#00ff88"
        ax_img.add_patch(patches.Rectangle(
            (x, y_pos), w, h, linewidth=3, edgecolor=c_box, facecolor="none"))
        ax_img.text(x, y_pos - 14, f"{emocion_es}  {confianza*100:.0f}%",
                    fontsize=13, color=c_box, fontweight="bold",
                    bbox=dict(facecolor="#0d1117", alpha=0.8, edgecolor=c_box))
    ax_img.set_title("Webcam -> Haar Cascade -> ViT", color="#aaaaaa", fontsize=11)

    labels_en  = [em_model.config.id2label[i] for i in range(len(todas_probs))]
    labels_vis = [EMOCIONES_ES.get(l, l) for l in labels_en]
    colores    = ["#00ff88" if l == emocion_en else "#2a2a4a" for l in labels_en]
    bars = ax_bars.barh(labels_vis, todas_probs * 100, color=colores, height=0.65)
    ax_bars.set_xlim(0, 105)
    ax_bars.set_xlabel("Confianza (%)", color="#888")
    ax_bars.set_title("IA EMOCIONAL - Probabilidades", color="#ff4466",
                      fontsize=10, fontweight="bold")
    ax_bars.set_facecolor("#161b22")
    ax_bars.tick_params(colors="white", labelsize=9)
    for spine in ax_bars.spines.values(): spine.set_color("#333")
    for bar, prob in zip(bars, todas_probs):
        ax_bars.text(prob * 100 + 1, bar.get_y() + bar.get_height() / 2,
                     f"{prob*100:.0f}%", va="center", color="white", fontsize=8)

    ax_text.set_facecolor("#161b22")
    ax_text.axis("off")
    ax_text.set_title("IA COGNITIVA - Respuesta", color="#4488ff",
                      fontsize=10, fontweight="bold", pad=8)
    wrapped = "\n".join(textwrap.wrap(respuesta, width=48))
    ax_text.text(0.05, 0.85, wrapped, transform=ax_text.transAxes,
                 va="top", ha="left", fontsize=9.5, color="#dddddd",
                 bbox=dict(facecolor="#0d1117", edgecolor="#4488ff",
                           alpha=0.8, boxstyle="round,pad=0.5"))

    fig.suptitle("IA Emocional + Cognitiva - Demo",
                 color="white", fontsize=15, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.savefig("demo_completo.png", dpi=100, bbox_inches="tight",
                facecolor="#0d1117")
    plt.show()
    plt.close(fig)

    print("\n" + "=" * 60)
    print("  RESULTADO FINAL")
    print("=" * 60)
    print(f"  Emocion   : {emocion_es}")
    print(f"  Confianza : {confianza*100:.1f}%")
    print(f"  Respuesta : {respuesta}")
    print("=" * 60)

demo_completo()


  DEMO COMPLETO - IA Emocional + Cognitiva

[1] Capturando imagen...
  Imagen capturada OK.
[2] IA Emocional - Analizando rostro...
  -> Neutral  (78.7%)
[3] IA Cognitiva - Generando respuesta...
  -> Respuesta OK.

  RESULTADO FINAL
  Emocion   : Neutral
  Confianza : 78.7%
  Respuesta : El usuario ha elegido el valor "Neutral" para su emocionamiento, lo que significa que no se puede determinar con exactitud si está disfrutando o sufriendo de alguna emoción específica. Esta valoración implica que el


**Conexión con lo que vieron en clase:**
- **CNN / ViT** → extrae features del rostro (como las CNNs, pero con atención global)  
- **Clasificación** → softmax sobre 7 clases (como un MLP clasificador)  
- **Transformers** → el LLM usa el mismo mecanismo de atención que vieron en teoría  
- **Transfer Learning** → ambos modelos son pre-entrenados y los usamos sin reentrenar
